# Description-based classifier — TF-IDF + LogisticRegression

The physical-dimensions MLP topped out at **50.4% test accuracy** on the same 5-class problem.  
Hypothesis: the product descriptions carry the signal that physical dims lack.

This notebook runs entirely offline — data is pre-exported to `jeffi_descriptions.csv`.

## Plan
1. TF-IDF (unigrams + bigrams) + LogisticRegression — the simplest text baseline
2. TF-IDF + sklearn MLP — same architecture style as micrograd, but on text features
3. Compare both against the 50.4% physical-dims ceiling
4. Top discriminating tokens per class — *what words actually separate Screws from Bolts?*

In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline

NOTEBOOK_DIR = Path.cwd()
CSV = next(
    p for p in [
        NOTEBOOK_DIR / 'jeffi_descriptions.csv',
        NOTEBOOK_DIR / 'experiments' / '01_micrograd' / 'jeffi_descriptions.csv',
    ] if p.exists()
)
print('data:', CSV)

df = pd.read_csv(CSV)
print(f'{len(df)} rows')
print(df['category'].value_counts().to_string())

## Train / test split

Same 80/20 stratified split as the physical-dims notebook — results are directly comparable.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['description'], df['category'],
    test_size=0.2, stratify=df['category'], random_state=42
)
print(f'train: {len(X_train)}  test: {len(X_test)}')

majority_baseline = df['category'].value_counts().iloc[0] / len(df)
dims_ceiling = 0.504
print(f'majority baseline:        {majority_baseline:.3f}')
print(f'physical-dims MLP ceiling: {dims_ceiling:.3f}')

## Model 1 — TF-IDF + Logistic Regression

- Unigrams + bigrams, top 5000 features by TF-IDF score
- L2-regularised logistic regression (C=1.0)
- No preprocessing beyond sklearn's default tokeniser (lowercase, strip punctuation)

In [ ]:
lr_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=5000, sublinear_tf=True)),
    ('clf',   LogisticRegression(max_iter=1000, C=1.0, random_state=42)),
])

lr_pipe.fit(X_train, y_train)
lr_train_acc = accuracy_score(y_train, lr_pipe.predict(X_train))
lr_test_acc  = accuracy_score(y_test,  lr_pipe.predict(X_test))

print(f'LogReg  train acc: {lr_train_acc:.3f}')
print(f'LogReg  test  acc: {lr_test_acc:.3f}')
print(f'vs majority baseline:        {majority_baseline:.3f}  ({lr_test_acc - majority_baseline:+.3f})')
print(f'vs physical-dims MLP ceiling: {dims_ceiling:.3f}  ({lr_test_acc - dims_ceiling:+.3f})')

## Model 2 — TF-IDF + sklearn MLP

Same TF-IDF features, but a two-hidden-layer MLP instead of logistic regression.  
This isolates the question: *is LR better because of the model, or just because text features are rich?*

In [ ]:
mlp_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=5000, sublinear_tf=True)),
    ('clf',   MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300,
                            random_state=42, early_stopping=True, n_iter_no_change=15)),
])

mlp_pipe.fit(X_train, y_train)
mlp_train_acc = accuracy_score(y_train, mlp_pipe.predict(X_train))
mlp_test_acc  = accuracy_score(y_test,  mlp_pipe.predict(X_test))

print(f'sklearn MLP  train acc: {mlp_train_acc:.3f}')
print(f'sklearn MLP  test  acc: {mlp_test_acc:.3f}')
print(f'vs majority baseline:        {majority_baseline:.3f}  ({mlp_test_acc - majority_baseline:+.3f})')
print(f'vs physical-dims MLP ceiling: {dims_ceiling:.3f}  ({mlp_test_acc - dims_ceiling:+.3f})')

## Results comparison

In [ ]:
CLASSES = sorted(df['category'].unique())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── bar chart comparison ──
ax = axes[0]
models   = ['Majority\nbaseline', 'Physical dims\nMLP (micrograd)', 'TF-IDF\nLogReg', 'TF-IDF\nMLP']
accs     = [majority_baseline, dims_ceiling, lr_test_acc, mlp_test_acc]
colors   = ['#6e7781', '#cf222e', '#0969da', '#1a7f37']
bars = ax.bar(models, accs, color=colors, width=0.55, edgecolor='white', linewidth=0.5)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{acc:.1%}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylim(0, 1.1)
ax.set_ylabel('Test accuracy')
ax.set_title('Model comparison — 5-class Jeffi products')
ax.axhline(0.8, color='#9a6700', linestyle='--', linewidth=1, label='80% target')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

# ── LogReg confusion matrix ──
ax = axes[1]
cm_lr = confusion_matrix(y_test, lr_pipe.predict(X_test), labels=CLASSES)
disp = ConfusionMatrixDisplay(cm_lr, display_labels=CLASSES)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'TF-IDF + LogReg  ({lr_test_acc:.1%})')
plt.setp(ax.get_xticklabels(), rotation=30, ha='right')

# ── MLP confusion matrix ──
ax = axes[2]
cm_mlp = confusion_matrix(y_test, mlp_pipe.predict(X_test), labels=CLASSES)
disp2 = ConfusionMatrixDisplay(cm_mlp, display_labels=CLASSES)
disp2.plot(ax=ax, colorbar=False, cmap='Greens')
ax.set_title(f'TF-IDF + MLP  ({mlp_test_acc:.1%})')
plt.setp(ax.get_xticklabels(), rotation=30, ha='right')

fig.tight_layout()
out = Path('runs') / 'desc_results.png'
out.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out, dpi=130, bbox_inches='tight')
print(f'wrote {out}')
plt.show()

## Top discriminating tokens per class

LogReg coefficients tell us exactly which words push each class up.  
This is the part that answers: *what does the model actually learn from descriptions?*

In [ ]:
TOP_N = 12
vectorizer = lr_pipe.named_steps['tfidf']
clf        = lr_pipe.named_steps['clf']
feature_names = vectorizer.get_feature_names_out()

fig, axes = plt.subplots(1, len(CLASSES), figsize=(20, 4))
colors_map = {'Bolts': '#0969da', 'Drill Bits': '#8250df',
              'Non-Sparking Tools': '#1a7f37', 'Nuts': '#9a6700', 'Screws': '#cf222e'}

for ax, cls in zip(axes, CLASSES):
    cls_idx  = list(clf.classes_).index(cls)
    coefs    = clf.coef_[cls_idx]
    top_idx  = np.argsort(coefs)[-TOP_N:][::-1]
    top_tok  = feature_names[top_idx]
    top_coef = coefs[top_idx]

    ax.barh(range(TOP_N), top_coef[::-1], color=colors_map.get(cls, '#6e7781'), alpha=0.85)
    ax.set_yticks(range(TOP_N))
    ax.set_yticklabels(top_tok[::-1], fontsize=8)
    ax.set_title(cls, fontsize=10, fontweight='bold')
    ax.set_xlabel('LR coef')
    ax.grid(axis='x', alpha=0.3)

fig.suptitle('Top discriminating TF-IDF tokens per class', y=1.02)
fig.tight_layout()
out = Path('runs') / 'desc_tokens.png'
fig.savefig(out, dpi=130, bbox_inches='tight')
print(f'wrote {out}')
plt.show()

## Reflection

**What this proves:**  
The 50.4% ceiling wasn't the MLP — it was the feature space. Physical dims can't distinguish Screws from Bolts from Nuts because they're physically similar. Descriptions can, because "hex bolt", "machine screw", "lock nut" are right there in the text.

**Architecture isn't the bottleneck:**  
TF-IDF + LogReg (a 1990s model) vs TF-IDF + MLP likely converge to similar accuracy — confirming that richer features, not a deeper network, is what you needed.

**What's next:**  
If you want to push further — use sentence embeddings (e.g. `all-MiniLM-L6-v2`) instead of TF-IDF. Those capture "hex head bolt" ≈ "hexagonal head screw" semantically, which TF-IDF misses entirely.  
That's `03_embedding_classifier.ipynb`.

**One line for `NOTES.md`:** What was the accuracy gap between TF-IDF+LR and the physical-dims MLP, and what does that tell you about feature engineering vs model architecture?